# Rosenstein LLE — Session 1 full-window PPS surrogate tests

Kiểm định toàn bộ window 60 giây Processed đã được đưa vào phân tích của Session 1. Với mỗi window, notebook ghép đúng `rho_star`, sinh đúng `M=39` PPS surrogate bằng `pps.py`, tính Rosenstein LLE với cấu hình của window gốc và thực hiện finite-sample two-sided rank test. `fit_r2` chỉ được lưu như diagnostic, không dùng để loại surrogate. Không sử dụng hoặc sửa `statistic_test.py`.


In [ ]:
# Cell 1 — Setup + frozen configuration
import hashlib
import json
import os
import sys
import time
from importlib import import_module
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed, parallel_config


def find_repo_root(start):
    for path in (start, *start.parents):
        if (path / "phase1" / "src").is_dir():
            return path
    raise FileNotFoundError("Repository root was not found.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

loader = import_module("phase1.src.dataloader.loader")
pps = import_module("phase1.src.surrogates.pps")
lyapunov = import_module("phase1.src.chaos.lyapunov")
load_segmented_session = loader.load_segmented_session
generate_pps_signal = pps.generate_pps_signal
compute_rosenstein_lle = lyapunov.compute_rosenstein_lle

SESSION = 1
SESSION_FILE = "sample_1.csv"
REPRESENTATION = "processed"
WINDOW_SIZES_S = (60,)
M = 39
ALPHA = 0.05
ALTERNATIVE = "two-sided"
MASTER_SEED = 20260828
MIN_INITIAL_PAIRS = 50
MIN_FIT_PAIRS = 30
MIN_R2 = 0.0  # Keep fit_r2 as a diagnostic; do not reject surrogate LLE by R².
N_JOBS = min(8, os.cpu_count() or 1)
FORCE_RECOMPUTE = False
STATE_NAMES = {0: "Awake", 1: "Drowsy"}
WINDOW_KEYS = ["session", "representation", "state", "window_size_s", "window_id"]

SEGMENTED_DATA_DIR = REPO_ROOT / "phase1" / "segmentated_data" / "dhdata"
RHO_PATH = (REPO_ROOT / "phase1" / "results" / "pps" /
            "pps_radius_calibration" / "csv" / "pps_radius_session_01.csv")
LLE_PATH = (REPO_ROOT / "phase1" / "results" / "lle" / "processed" /
            "session_01_processed_rosenstein_lle.csv")
OUTPUT_DIR = REPO_ROOT / "phase1" / "results" / "lle" / "surrogates" / "processed"
WINDOW_OUTPUT = OUTPUT_DIR / "session_01_60s_processed_lle_pps_rank_test.csv"
SURROGATE_OUTPUT = OUTPUT_DIR / "session_01_60s_processed_lle_pps_surrogates.csv"

assert M == 39 and ALTERNATIVE == "two-sided"
assert np.isclose(2 / (M + 1), ALPHA)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
print(f"Session={SESSION}; Processed; sizes={WINDOW_SIZES_S}; M={M}; n_jobs={N_JOBS}")
print(f"Minimum attainable two-sided p-value: {2 / (M + 1):.3f}")


In [ ]:
# Cell 2 — Load all included Processed windows and match rho/LLE rows
rho_table = pd.read_csv(RHO_PATH)
rho_table = rho_table.loc[
    rho_table["session"].eq(SESSION)
    & rho_table["representation"].str.casefold().eq(REPRESENTATION)
    & rho_table["window_size"].isin(WINDOW_SIZES_S)
].copy()
lle_table = pd.read_csv(LLE_PATH)
lle_table = lle_table.loc[
    lle_table["session"].eq(SESSION)
    & lle_table["representation"].str.casefold().eq(REPRESENTATION)
    & lle_table["window_size_s"].isin(WINDOW_SIZES_S)
    & lle_table["analysis_included"].astype(bool)
].copy()

batches, segmented_metadata = load_segmented_session(
    SESSION_FILE, data_dir=SEGMENTED_DATA_DIR,
    window_sizes=WINDOW_SIZES_S, representation=REPRESENTATION,
    stationarity_only=True,
)
window_tasks = []
integrity_rows = []

for duration_s in WINDOW_SIZES_S:
    batch = batches[duration_s]
    for index, window_id in enumerate(batch["window_id"]):
        state = STATE_NAMES[int(batch["label"][index])]
        identity = {
            "session": SESSION, "representation": REPRESENTATION,
            "state": state, "window_size_s": int(duration_s),
            "window_id": int(window_id),
        }
        rho_match = rho_table.loc[
            rho_table["window_size"].eq(duration_s)
            & rho_table["window_id"].eq(window_id)
            & rho_table["state"].str.casefold().eq(state.casefold())
        ]
        lle_match = lle_table.loc[
            lle_table["window_size_s"].eq(duration_s)
            & lle_table["window_id"].eq(window_id)
            & lle_table["state"].str.casefold().eq(state.casefold())
        ]
        assert len(rho_match) == len(lle_match) == 1
        signal = np.asarray(batch["signal"][index], dtype=float)
        assert signal.ndim == 1 and np.all(np.isfinite(signal))
        assert signal.size == round(duration_s * float(batch["fs"]))
        task = {
            **identity, "signal": signal,
            "sampling_rate": float(batch["fs"]),
            "rho_star": float(rho_match.iloc[0]["rho_star"]),
            "lle_reference": lle_match.iloc[0].to_dict(),
        }
        window_tasks.append(task)
        integrity_rows.append({
            **identity, "sampling_rate_hz": task["sampling_rate"],
            "n_samples": signal.size, "rho_star": task["rho_star"],
            "original_lle_valid": bool(lle_match.iloc[0]["valid"]),
        })

integrity_table = pd.DataFrame(integrity_rows).sort_values(WINDOW_KEYS).reset_index(drop=True)
assert len(window_tasks) == len(integrity_table) == len(rho_table) == len(lle_table) == 31
assert integrity_table["original_lle_valid"].all()
assert not integrity_table.duplicated(WINDOW_KEYS).any()
display(
    integrity_table.groupby(["window_size_s", "state"], as_index=False)
    .agg(n_windows=("window_id", "size"), median_rho=("rho_star", "median"))
    .round(6)
)


In [ ]:
# Cell 3 — Deterministic seeds, rank test, and per-window evaluator
def derive_window_seed(identity):
    payload = json.dumps(
        {"master_seed": MASTER_SEED, **identity},
        sort_keys=True, separators=(",", ":"),
    ).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little")


def derive_surrogate_seeds(window_seed):
    return [
        int.from_bytes(
            hashlib.sha256(f"{window_seed}:{surrogate_id}".encode("ascii")).digest()[:8],
            "little",
        )
        for surrogate_id in range(1, M + 1)
    ]


def two_sided_rank_test(original, surrogate_values, alpha=ALPHA):
    values = np.asarray(surrogate_values, dtype=float)
    if values.ndim != 1 or values.size == 0 or not np.all(np.isfinite(values)):
        raise ValueError("Surrogate statistics must be a finite non-empty 1D array.")
    n_lower = int(np.count_nonzero(values <= original))
    n_upper = int(np.count_nonzero(values >= original))
    n_equal = int(np.count_nonzero(values == original))
    n_less = int(np.count_nonzero(values < original))
    denominator = values.size + 1
    p_lower = (n_lower + 1) / denominator
    p_upper = (n_upper + 1) / denominator
    p_value = min(1.0, 2.0 * min(p_lower, p_upper))
    reject = bool(p_value <= alpha)
    direction = "none"
    if reject and p_upper < p_lower:
        direction = "higher"
    elif reject and p_lower < p_upper:
        direction = "lower"
    return {
        "rank_ascending": float(1 + n_less + 0.5 * n_equal),
        "n_surrogates_le_original": n_lower,
        "n_surrogates_ge_original": n_upper,
        "p_lower": float(p_lower), "p_upper": float(p_upper),
        "p_two_sided": float(p_value), "reject_h0": reject,
        "direction": direction,
    }


def evaluate_window(task):
    identity = {name: task[name] for name in WINDOW_KEYS}
    reference = task["lle_reference"]
    config = {
        "sampling_rate": task["sampling_rate"],
        "m": int(reference["m"]),
        "tau_samples": int(reference["tau_samples"]),
        "fit_start_s": float(reference["fit_start_s"]),
        "fit_end_s": float(reference["fit_end_s"]),
        "max_follow_s": float(reference["max_follow_s"]),
        "theiler_s": float(reference["theiler_s"]),
        "min_initial_pairs": MIN_INITIAL_PAIRS,
        "min_fit_pairs": MIN_FIT_PAIRS, "min_r2": MIN_R2,
    }
    original = compute_rosenstein_lle(task["signal"], **config)
    if not original.valid or not np.isclose(
        original.lle, float(reference["lle_1_per_s"]), rtol=0, atol=1e-12
    ):
        raise RuntimeError(f"Original LLE integrity failure: {identity}")

    window_seed = derive_window_seed(identity)
    surrogate_rows, values = [], []
    for surrogate_id, seed in enumerate(derive_surrogate_seeds(window_seed), start=1):
        surrogate = generate_pps_signal(
            task["signal"], tau=config["tau_samples"], m=config["m"],
            rho=task["rho_star"], rng=np.random.default_rng(seed),
        )
        result = compute_rosenstein_lle(surrogate, **config)
        if not np.isfinite(result.lle):
            raise RuntimeError(f"Non-finite surrogate LLE: {identity}, id={surrogate_id}")
        values.append(result.lle)
        surrogate_rows.append({
            **identity, "surrogate_id": surrogate_id, "seed": seed,
            "rho_star": task["rho_star"], "lle_1_per_s": result.lle,
            "fit_r2": result.fit_r2, "n_pairs_initial": result.n_pairs_initial,
            "n_pairs_fit_min": result.n_pairs_fit_min,
            "valid": result.valid, "qc_reason": result.qc_reason,
        })

    values = np.asarray(values)
    test = two_sided_rank_test(original.lle, values)
    valid_values = np.asarray([row["lle_1_per_s"] for row in surrogate_rows if row["valid"]])
    sensitivity = two_sided_rank_test(original.lle, valid_values) if valid_values.size else None
    window_row = {
        **identity, "sampling_rate_hz": task["sampling_rate"],
        "n_samples": task["signal"].size, "rho_star": task["rho_star"],
        "m": config["m"], "tau_samples": config["tau_samples"],
        "theiler_samples": original.theiler_samples,
        "fit_start_s": config["fit_start_s"], "fit_end_s": config["fit_end_s"],
        "M": M, "original_lle_1_per_s": original.lle,
        "original_fit_r2": original.fit_r2,
        "surrogate_lle_mean": float(np.mean(values)),
        "surrogate_lle_median": float(np.median(values)),
        "surrogate_lle_sd": float(np.std(values, ddof=1)),
        "surrogate_lle_min": float(np.min(values)),
        "surrogate_lle_max": float(np.max(values)),
        "n_surrogate_valid": int(valid_values.size),
        "n_surrogate_fit_r2_below_0_90": int(sum(row["fit_r2"] < 0.90 for row in surrogate_rows)),
        **test, "qc_sensitivity_M": int(valid_values.size),
        "qc_sensitivity_p_two_sided": sensitivity["p_two_sided"] if sensitivity else np.nan,
        "qc_sensitivity_reject_h0": sensitivity["reject_h0"] if sensitivity else False,
        "master_seed": MASTER_SEED, "window_seed": window_seed,
    }
    return window_row, surrogate_rows

print("Local PPS/LLE/rank-test functions are ready.")


In [ ]:
# Cell 4 — Run all windows or load the validated cache
def cache_is_complete(window_path, surrogate_path):
    if not window_path.is_file() or not surrogate_path.is_file():
        return False
    cached_windows = pd.read_csv(window_path)
    cached_surrogates = pd.read_csv(surrogate_path)
    expected = set(map(tuple, integrity_table[WINDOW_KEYS].itertuples(index=False, name=None)))
    observed = set(map(tuple, cached_windows[WINDOW_KEYS].itertuples(index=False, name=None)))
    counts = cached_surrogates.groupby(WINDOW_KEYS).size()
    return (
        len(cached_windows) == 31 and len(cached_surrogates) == 31 * M
        and observed == expected and len(counts) == 31 and counts.eq(M).all()
        and cached_windows["M"].eq(M).all()
        and cached_windows["master_seed"].eq(MASTER_SEED).all()
    )


if not FORCE_RECOMPUTE and cache_is_complete(WINDOW_OUTPUT, SURROGATE_OUTPUT):
    window_results = pd.read_csv(WINDOW_OUTPUT)
    surrogate_results = pd.read_csv(SURROGATE_OUTPUT)
    print("Loaded complete cached results. Set FORCE_RECOMPUTE=True to regenerate.")
else:
    started = time.perf_counter()
    with parallel_config(backend="loky", inner_max_num_threads=1):
        bundles = Parallel(n_jobs=N_JOBS, verbose=10)(
            delayed(evaluate_window)(task) for task in window_tasks
        )
    window_results = pd.DataFrame([bundle[0] for bundle in bundles])
    surrogate_results = pd.DataFrame([row for bundle in bundles for row in bundle[1]])
    window_results = window_results.sort_values(WINDOW_KEYS).reset_index(drop=True)
    surrogate_results = surrogate_results.sort_values(WINDOW_KEYS + ["surrogate_id"]).reset_index(drop=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    window_results.to_csv(WINDOW_OUTPUT, index=False)
    surrogate_results.to_csv(SURROGATE_OUTPUT, index=False)
    print(f"Computed all windows in {(time.perf_counter() - started) / 60:.2f} min")

assert cache_is_complete(WINDOW_OUTPUT, SURROGATE_OUTPUT)
assert len(window_results) == 31 and len(surrogate_results) == 1209
assert np.isfinite(window_results["p_two_sided"]).all()
assert np.isfinite(surrogate_results["lle_1_per_s"]).all()
print(f"Window output: {WINDOW_OUTPUT}")
print(f"Surrogate audit output: {SURROGATE_OUTPUT}")


In [ ]:
# Cell 5 — Final rank-test results (fit_r2 is diagnostic only)
window_results = window_results.copy()
window_results["lle_gap_vs_surrogate_median"] = (
    window_results["original_lle_1_per_s"] - window_results["surrogate_lle_median"]
)
if "n_surrogate_fit_r2_below_0_90" not in window_results:
    diagnostic_low_r2 = (
        surrogate_results.assign(fit_r2_below_0_90=surrogate_results["fit_r2"].lt(0.90))
        .groupby(WINDOW_KEYS, as_index=False)["fit_r2_below_0_90"].sum()
        .rename(columns={"fit_r2_below_0_90": "n_surrogate_fit_r2_below_0_90"})
    )
    window_results = window_results.merge(diagnostic_low_r2, on=WINDOW_KEYS, validate="one_to_one")

# All finite slopes enter the rank test; fit_r2 does not change validity.
assert window_results["n_surrogate_valid"].eq(M).all()
assert surrogate_results["valid"].all()
assert surrogate_results["qc_reason"].eq("ok").all()
assert window_results["reject_h0"].all()
assert window_results["direction"].eq("higher").all()
assert (window_results["original_lle_1_per_s"] > window_results["surrogate_lle_max"]).all()
assert int(window_results["n_surrogate_fit_r2_below_0_90"].sum()) == 204

result_overview = pd.DataFrame([{
    "windows": len(window_results),
    "surrogates": len(surrogate_results),
    "rank_rejections": int(window_results["reject_h0"].sum()),
    "finite_valid_surrogate_LLE": int(surrogate_results["valid"].sum()),
    "diagnostic_fit_R2_below_0_90": int(window_results["n_surrogate_fit_r2_below_0_90"].sum()),
}])
display(result_overview)
display(window_results[
    WINDOW_KEYS + ["original_lle_1_per_s", "surrogate_lle_median",
                   "n_surrogate_fit_r2_below_0_90", "rank_ascending",
                   "p_two_sided", "reject_h0", "direction"]
].round(6))


In [ ]:
# Cell 6 — State summaries and diagnostic figures
group_summary = (
    window_results.groupby(["window_size_s", "state"], as_index=False)
    .agg(
        n_windows=("window_id", "size"),
        rank_reject=("reject_h0", "sum"),
        median_original_LLE=("original_lle_1_per_s", "median"),
        median_surrogate_LLE=("surrogate_lle_median", "median"),
        median_LLE_gap=("lle_gap_vs_surrogate_median", "median"),
        fit_R2_below_0_90=("n_surrogate_fit_r2_below_0_90", "sum"),
    )
)
display(group_summary.round(6))

colors = {"Awake": "#0072B2", "Drowsy": "#D55E00"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for state, state_rows in window_results.groupby("state"):
    axes[0].scatter(
        state_rows["surrogate_lle_median"], state_rows["original_lle_1_per_s"],
        color=colors[state], s=45, alpha=0.8, label=state,
    )
limits = [0.50, 1.08]
axes[0].plot(limits, limits, color="grey", ls="--", lw=1)
axes[0].set(title="Original vs median PPS LLE", xlabel="Median PPS LLE [1/s]",
            ylabel="Original LLE [1/s]", xlim=limits, ylim=limits)
axes[0].legend(frameon=False)
axes[1].hist(surrogate_results["fit_r2"], bins=30, color="#0072B2", alpha=0.8)
axes[1].axvline(0.90, color="#D55E00", ls="--", label="R²=0.90 (diagnostic only)")
axes[1].set(title="Surrogate fit R² distribution", xlabel="Fit R²", ylabel="Count")
axes[1].legend(frameon=False)
fig.tight_layout()
plt.show()

print("No Awake–Drowsy inference is performed: windows overlap and belong to one session.")


# Session 1 — 60-s Processed LLE–PPS Surrogate Report

## Configuration and cohort

- Session: **1**; representation: **Processed**
- Included windows: **31 window 60 s** — 24 Awake và 7 Drowsy
- PPS ensemble: `M=39` per window; total **1.209 surrogate signals**
- `rho_star`: matched by session, representation, state, duration and `window_id`
- Test statistic: Rosenstein LLE với cấu hình frozen của từng original window
- `fit_r2` được lưu như diagnostic nhưng **không dùng để loại surrogate LLE** (`min_r2=0`)
- Test: two-sided finite-sample rank test with `+1` correction, `alpha=0.05`
- Reproducibility: master seed `20260828`; deterministic identity-derived window/surrogate seeds

## Numerical rank-test result

Original LLE cao hơn toàn bộ 39 surrogate LLE trong **31/31 windows**. Vì vậy mọi kiểm định đều có ascending rank `40/40`, hướng `higher`, và `p=2/(39+1)=0.05`. Median original LLE là **0.92046 1/s**; median của toàn bộ 1.209 surrogate LLE là **0.59184 1/s**. Median chênh lệch trong từng window giữa original và surrogate median là **0.33636 1/s** (range **0.19513–0.45788 1/s**).

| State | Windows | Rank-test reject | Median original LLE | Median PPS LLE | Median gap |
|---|---:|---:|---:|---:|---:|
| Awake | 24 | 24 | 0.93638 | 0.59306 | 0.34175 |
| Drowsy | 7 | 7 | 0.85085 | 0.59654 | 0.25653 |

## Fit-R² diagnostic

Có **204/1.209 (16,87%)** surrogate fit có `R²<0.90`, nhưng mọi slope đều hữu hạn, đủ pair support và thấp hơn original tương ứng. Sau khi bỏ `fit_r2` khỏi tiêu chí invalidation, toàn bộ **1.209/1.209** surrogate có `valid=True`, `qc_reason=ok`, nên cả 31 window giữ đủ `M=39`. `fit_r2` vẫn nằm trong CSV để audit đường divergence nhưng không làm thay đổi rank test.

## Interpretation

PPS null ensemble tạo LLE thấp hơn tín hiệu gốc một cách nhất quán ở toàn bộ 31 window 60 giây. Experiment **PASS về triển khai và directional separation** theo chính sách không dùng `fit_r2` để loại surrogate. Tuy nhiên, `p=0.05` nằm đúng giới hạn phân giải nhỏ nhất của two-sided test với `M=39`, nên không nên diễn giải là bằng chứng mạnh hơn mức biên này.

Không dùng 31 window chồng lấp trong cùng một session để suy luận thống kê Awake–Drowsy. Original LLE cao hơn PPS surrogates cũng không tự thân chứng minh deterministic chaos.

## Recommended next step

Nếu cần p-value nhỏ hơn 0.05 hoặc độ phân giải tốt hơn, tăng số surrogate theo kế hoạch định trước trước khi chạy full dataset; tiếp tục giữ `fit_r2` như diagnostic thay vì tiêu chí loại surrogate.
